# Phân Khúc Khách Hàng Bằng Thuật Toán K-Means & PySpark
---
Sổ tay nghiên cứu (Jupyter Notebook) thực hiện phân nhóm khách hàng dựa trên mô hình RFM sử dụng thư viện học máy song song PySpark MLlib và trực quan hóa kết quả.

In [6]:
%matplotlib inline
import os
import sys

# Force UTF-8 encoding for stdout/stderr to avoid UnicodeEncodeError on Windows
if sys.stdout.encoding != 'utf-8' and hasattr(sys.stdout, 'reconfigure'):
    sys.stdout.reconfigure(encoding='utf-8')
if sys.stderr.encoding != 'utf-8' and hasattr(sys.stderr, 'reconfigure'):
    sys.stderr.reconfigure(encoding='utf-8')

# ======================================================
# TỰ ĐỘNG KIỂM TRA VÀ CÀI ĐẶT THƯ VIỆN NẾU THIẾU
# ======================================================
def check_and_install_packages():
    """Kiểm tra và cài đặt tự động các thư viện bổ sung nếu thiếu."""
    required_packages = ["numpy", "pandas", "matplotlib", "scikit-learn"]
    missing_packages = []

    for package in required_packages:
        import_name = "sklearn" if package == "scikit-learn" else package
        try:
            __import__(import_name)
        except ImportError:
            missing_packages.append(package)

    if missing_packages:
        print(f"-> Phát hiện thiếu thư viện: {missing_packages}. Đang tự động cài đặt qua pip...")
        try:
            import subprocess
            subprocess.check_call([sys.executable, "-m", "pip", "install", *missing_packages])
            print("-> Đã cài đặt thành công!\n")
        except Exception as e:
            print(f"-> Không thể tự động cài đặt thư viện: {str(e)}")
            print("-> Vui lòng kết nối Internet hoặc tự cài thủ công bằng lệnh: pip install " + " ".join(missing_packages))

# Chạy trình cài đặt tự động trước khi import các thư viện khoa học dữ liệu
check_and_install_packages()

# Tiến hành import sau khi đã đảm bảo các thư viện tồn tại
try:
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    import matplotlib.cm as cm
    import matplotlib.colors as mcolors
    from sklearn.metrics import silhouette_samples, silhouette_score
    HAS_VISUAL_LIBS = True
except ImportError:
    HAS_VISUAL_LIBS = False

# Thiết lập Python cho PySpark trên môi trường Windows
os.environ["PYSPARK_PYTHON"] = "python"
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

from pyspark.sql import SparkSession
from pyspark.sql.types import *
from pyspark.sql.functions import (
    col,
    count,
    avg,
    round as spark_round,
    sum as spark_sum,
    when,
    min as spark_min,
    max as spark_max,
    countDistinct,
    datediff,
    lit,
    log1p
)
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import ClusteringEvaluator
from pyspark.ml import Pipeline

# ======================================================
# CÁC HÀM XỬ LÝ CHÍNH (SPARK PIPELINE & MODELING)
# ======================================================


print("-> Đã import các thư viện và kiểm tra môi trường thành công!")


-> Đã import các thư viện và kiểm tra môi trường thành công!


## Bước 1: Định nghĩa & Khởi tạo Spark Session
Thiết lập Spark Session với các cấu hình tối ưu chạy trên môi trường local.

In [7]:
def create_spark_session(app_name="Nhom10_Spark_Read_HDFS_Viet", master="spark://26.97.56.101:7077"):
    """Khởi tạo Spark Session với cấu hình kết nối Spark Master và HDFS."""
    DRIVER_IP = "26.155.115.30"
    spark = (
        SparkSession.builder
        .appName(app_name)
        .master(master)
        .config("spark.executor.memory", "1g")
        .config("spark.executor.cores", "1")
        .config("spark.cores.max", "2")
        .config("spark.driver.host", DRIVER_IP)
        .config("spark.driver.bindAddress", "0.0.0.0")
        .config("spark.dynamicAllocation.enabled", "false")
        .getOrCreate()
    )
    # Tối ưu hóa số lượng shuffle partitions cho cụm
    spark.conf.set("spark.sql.shuffle.partitions", "8")
    spark.sparkContext.setLogLevel("WARN")
    return spark



# Khởi tạo Spark Session kết nối với Cluster
spark = create_spark_session()
print("-> Khởi tạo Spark Session kết nối HDFS & Spark Master thành công!")


-> Khởi tạo Spark Session kết nối HDFS & Spark Master thành công!


## Bước 2: Đọc dữ liệu từ File CSV
Hàm xử lý đọc dữ liệu từ file CSV cục bộ.

In [8]:
def load_and_clean_data(spark, data_path):
    """Đọc dữ liệu từ file CSV HDFS."""
    schema = StructType([
        StructField("Category", StringType(), True),
        StructField("City", StringType(), True),
        StructField("Country", StringType(), True),
        StructField("Customer_ID", StringType(), True),
        StructField("Customer_Name", StringType(), True),
        StructField("Discount", DoubleType(), True),
        StructField("Market", StringType(), True),
        StructField("Order_Date", DateType(), True),
        StructField("Order_ID", StringType(), True),
        StructField("Order_Priority", StringType(), True),
        StructField("Product_ID", StringType(), True),
        StructField("Product_Name", StringType(), True),
        StructField("Profit", DoubleType(), True),
        StructField("Quantity", IntegerType(), True),
        StructField("Region", StringType(), True),
        StructField("Sales", DoubleType(), True),
        StructField("Segment", StringType(), True),
        StructField("Ship_Date", DateType(), True),
        StructField("Ship_Mode", StringType(), True),
        StructField("Shipping_Cost", DoubleType(), True),
        StructField("State", StringType(), True),
        StructField("Sub_Category", StringType(), True),
        StructField("Year", IntegerType(), True),
        StructField("weeknum", IntegerType(), True)
    ])

    df_ml = (
        spark.read
        .option("header", True)
        .option("dateFormat", "yyyy-MM-dd")
        .option("quote", "\"")
        .option("escape", "\"")
        .option("multiLine", True)
        .schema(schema)
        .csv(data_path)
    )
    return df_ml



# Cấu hình đường dẫn dữ liệu trên HDFS
DATA_PATH = "hdfs://26.97.56.101:9000/bigdata/superstore/input/G10_dataset.csv"
BEST_K = 3

# Tiến hành đọc và làm sạch dữ liệu từ HDFS
df_ml = load_and_clean_data(spark, DATA_PATH)
print(f"-> Đọc dữ liệu thành công!")
print(f"   Tổng số dòng: {df_ml.count()}")
print(f"   Tổng số cột: {len(df_ml.columns)}")


-> Đọc dữ liệu thành công!
   Tổng số dòng: 500000
   Tổng số cột: 24


### Xem 5 dòng dữ liệu đầu tiên dưới dạng DataFrame Pandas (hiển thị bảng đẹp hơn trên Notebook)

In [4]:
df_ml.limit(5).toPandas()


,Category,City,Country,Customer_ID,Customer_Name,Discount,Market,Order_Date,Order_ID,Order_Priority,...,Region,Sales,Segment,Ship_Date,Ship_Mode,Shipping_Cost,State,Sub_Category,Year,weeknum
0,Office Supplies,Los Angeles,United States,LS-172304,Lycoris Saunders,0.0,US,2011-01-07,CA-2011-130813,High,...,West,19.0,Consumer,2011-01-09,Second Class,4.37,California,Paper,2011,2
1,Office Supplies,Los Angeles,United States,MV-174854,Mark Van Huff,0.0,US,2011-01-21,CA-2011-148614,Medium,...,West,19.0,Consumer,2011-01-26,Standard Class,0.94,California,Paper,2011,4
2,Office Supplies,Los Angeles,United States,CS-121304,Chad Sievert,0.0,US,2011-08-05,CA-2011-118962,Medium,...,West,21.0,Consumer,2011-08-09,Standard Class,1.81,California,Paper,2011,32
3,Office Supplies,Los Angeles,United States,CS-121304,Chad Sievert,0.0,US,2011-08-05,CA-2011-118962,Medium,...,West,111.0,Consumer,2011-08-09,Standard Class,4.59,California,Paper,2011,32
4,Office Supplies,Los Angeles,United States,AP-109154,Arthur Prichep,0.0,US,2011-09-29,CA-2011-146969,High,...,West,6.0,Consumer,2011-10-03,Standard Class,1.32,California,Paper,2011,40


## Bước 3: Tính toán các chỉ số RFM & Tiền xử lý Outliers
Gom nhóm theo khách hàng, tính toán các chỉ số: 
- **Recency (R)**: Khoảng thời gian từ lần mua cuối cùng.
- **Frequency (F)**: Tần suất mua hàng.
- **Monetary (M)**: Tổng số tiền chi tiêu.

Sau đó tiến hành làm sạch các giá trị khuyết thiếu (Missing Values), khử ngoại lai bằng phương pháp Clipping (1% - 99%), và chuẩn hóa phân phối bằng phép biến đổi log (`log1p`).

In [9]:
def build_rfm_features(df_ml):
    """Tính toán RFM, xử lý khuyết thiếu, loại bỏ ngoại lai bằng Clipping (1%-99%), và Log1p."""
    print("\n===== TIẾN HÀNH GOM NHÓM DỮ LIỆU THEO CUSTOMER (RFM) =====")
    
    # Tính ngày mua hàng cuối cùng của hệ thống làm mốc tính Recency
    max_order_date = df_ml.select(spark_max("Order_Date")).first()[0]
    print("Ngày đặt hàng gần nhất trong hệ thống (Mốc tính Recency):", max_order_date)

    # Gom nhóm theo Customer_ID và Customer_Name
    rfm_raw_df = (
        df_ml.groupBy("Customer_ID", "Customer_Name")
        .agg(
            datediff(lit(max_order_date), spark_max("Order_Date")).cast("double").alias("Recency"),
            countDistinct("Order_ID").cast("double").alias("Frequency"),
            spark_sum("Sales").alias("Monetary"),
            avg("Profit").alias("Avg_Profit"),
            avg("Discount").alias("Avg_Discount"),
            spark_sum("Quantity").cast("double").alias("Total_Quantity")
        )
    )

    print("\nCấu trúc bảng RFM ban đầu:")
    rfm_raw_df.show(5, truncate=False)

    print("\n===== XỬ LÝ MISSING VALUE =====")
    cluster_feature_cols_raw = ["Recency", "Frequency", "Monetary", "Avg_Profit", "Avg_Discount", "Total_Quantity"]

    null_check = rfm_raw_df.select([
        spark_sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
        for c in cluster_feature_cols_raw
    ])
    print("Số lượng missing values trong từng cột:")
    null_check.show(truncate=False)

    # Loại bỏ các dòng khuyết thiếu trong cột đặc trưng
    rfm_clean_df = rfm_raw_df.dropna(subset=cluster_feature_cols_raw)
    total_customers = rfm_clean_df.count()
    print(f"Tổng số khách hàng sau khi làm sạch: {total_customers}")

    print("\n===== XỬ LÝ OUTLIERS BẰNG PHƯƠNG PHÁP CLIPPING (1% - 99%) =====")
    rfm_clipped_df = rfm_clean_df
    for c in cluster_feature_cols_raw:
        # Tính toán percentile 1% và 99%
        quantiles = rfm_clean_df.approxQuantile(c, [0.01, 0.99], 0.01)
        lower_bound, upper_bound = quantiles[0], quantiles[1]
        print(f"Cột {c:15} | Lower Bound (1%): {lower_bound:10.4f} | Upper Bound (99%): {upper_bound:10.4f}")
        
        # Clip giá trị ngoài khoảng bounds về chính giá trị biên
        rfm_clipped_df = rfm_clipped_df.withColumn(
            c,
            when(col(c) < lower_bound, lower_bound)
            .when(col(c) > upper_bound, upper_bound)
            .otherwise(col(c))
        )

    print("\nThống kê mô tả sau khi xử lý Outlier (Clipped):")
    rfm_clipped_df.describe(cluster_feature_cols_raw).show(truncate=False)

    print("\n===== LOG1P TRANSFORMATION CHO RECENCY, FREQUENCY, MONETARY =====")
    rfm_transformed_df = (
        rfm_clipped_df
        .withColumn("log_Recency", log1p(col("Recency")))
        .withColumn("log_Frequency", log1p(col("Frequency")))
        .withColumn("log_Monetary", log1p(col("Monetary")))
    )
    return rfm_transformed_df, total_customers



# Tính toán RFM và tiền xử lý
rfm_transformed_df, total_customers = build_rfm_features(df_ml)
print(f"-> Tính toán RFM và tiền xử lý hoàn tất! Tổng số khách hàng: {total_customers}")

# Hiển thị mẫu 5 dòng dữ liệu sau biến đổi (bao gồm các cột log_Recency, log_Frequency, log_Monetary)
rfm_transformed_df.limit(5).toPandas()



===== TIẾN HÀNH GOM NHÓM DỮ LIỆU THEO CUSTOMER (RFM) =====
Ngày đặt hàng gần nhất trong hệ thống (Mốc tính Recency): 2014-12-31

Cấu trúc bảng RFM ban đầu:
+-----------+-------------------+-------+---------+--------+------------------+-------------------+--------------+
|Customer_ID|Customer_Name      |Recency|Frequency|Monetary|Avg_Profit        |Avg_Discount       |Total_Quantity|
+-----------+-------------------+-------+---------+--------+------------------+-------------------+--------------+
|RS-197654  |Roland Schwarz     |36.0   |8.0      |4160.0  |86.17036428571429 |0.1142857142857143 |54.0          |
|MS-173654  |Maribeth Schnelling|273.0  |10.0     |7445.0  |35.20564583333333 |0.16041666666666668|105.0         |
|NP-183254  |Naresj Patel       |25.0   |6.0      |5532.0  |57.566142857142864|0.05714285714285715|77.0          |
|GA-145154  |George Ashbrook    |149.0  |8.0      |3922.0  |60.064214285714286|0.0642857142857143 |48.0          |
|PJ-188354  |Patrick Jones      |39.0 

,Customer_ID,Customer_Name,Recency,Frequency,Monetary,Avg_Profit,Avg_Discount,Total_Quantity,log_Recency,log_Frequency,log_Monetary
0,RS-197654,Roland Schwarz,36.0,8.0,4160.0,86.170364,0.114286,54.0,3.610918,2.197225,8.333511
1,MS-173654,Maribeth Schnelling,273.0,10.0,7445.0,35.205646,0.160417,105.0,5.613128,2.397895,8.915432
2,NP-183254,Naresj Patel,25.0,6.0,5532.0,57.566143,0.057143,77.0,3.258097,1.945910,8.618485
3,GA-145154,George Ashbrook,149.0,8.0,3922.0,60.064214,0.064286,48.0,5.010635,2.197225,8.274612
4,PJ-188354,Patrick Jones,39.0,8.0,1220.0,34.010423,0.076923,46.0,3.688879,2.197225,7.107425


## Bước 4: Chuẩn hóa các thuộc tính đặc trưng bằng StandardScaler Pipeline
Đóng gói đặc trưng vào Vector và chuẩn hóa dữ liệu về cùng thang đo (Mean=0, Std=1).

In [10]:
def prepare_features_pipeline(rfm_transformed_df):
    """Đưa các đặc trưng vào VectorAssembler và chuẩn hóa bằng StandardScaler."""
    cluster_feature_cols = ["log_Recency", "log_Frequency", "log_Monetary"]

    assembler = VectorAssembler(
        inputCols=cluster_feature_cols,
        outputCol="raw_features"
    )

    scaler = StandardScaler(
        inputCol="raw_features",
        outputCol="features",
        withStd=True,
        withMean=True
    )

    pipeline_base = Pipeline(stages=[assembler, scaler])
    cluster_prepared = pipeline_base.fit(rfm_transformed_df).transform(rfm_transformed_df)
    cluster_prepared = cluster_prepared.cache()

    print("\n===== MẪU DỮ LIỆU SAU KHI PIPELINE SCALE FEATURES =====")
    cluster_prepared.select("Customer_ID", "features").show(5, truncate=False)
    return cluster_prepared



# Chuẩn bị Pipeline và scale đặc trưng
cluster_prepared = prepare_features_pipeline(rfm_transformed_df)
print("-> Trích xuất & Chuẩn hóa đặc trưng (Pipeline) hoàn tất!")



===== MẪU DỮ LIỆU SAU KHI PIPELINE SCALE FEATURES =====
+-----------+-------------------------------------------------------------+
|Customer_ID|features                                                     |
+-----------+-------------------------------------------------------------+
|RS-197654  |[-1.4079080031336098,0.4442333802535576,0.43659426321883965] |
|MS-173654  |[0.4070569545780417,0.840452895356058,0.9446061323696933]    |
|NP-183254  |[-1.7277337803036648,-0.051980980704313305,0.685374443930536]|
|GA-145154  |[-0.13909116962968363,0.4442333802535576,0.3851762141688444] |
|PJ-188354  |[-1.3372373681229415,0.4442333802535576,-0.6337662635156461] |
+-----------+-------------------------------------------------------------+
only showing top 5 rows
-> Trích xuất & Chuẩn hóa đặc trưng (Pipeline) hoàn tất!


## Bước 5: Thử nghiệm huấn luyện với các giá trị K để tìm số lượng cụm tối ưu
Chạy thuật toán K-Means cho $K$ chạy từ 2 đến 8 để thu thập điểm Silhouette và chi phí huấn luyện (Elbow Method).

In [12]:
def run_kmeans_optimization(cluster_prepared):
    """Thử nghiệm nhiều giá trị K khác nhau, thu thập chỉ số Silhouette và mẫu dữ liệu đầy đủ."""
    evaluator = ClusteringEvaluator(
        featuresCol="features",
        predictionCol="prediction",
        metricName="silhouette"
    )

    k_results = []
    k_models = {}
    samples_by_k = {}

    print("\n===== THỬ NHIỀU GIÁ TRỊ K =====")

    for k in range(2, 9):
        kmeans = KMeans(
            k=k,
            seed=10,
            featuresCol="features",
            predictionCol="prediction",
            maxIter=50
        )

        model = kmeans.fit(cluster_prepared)
        k_models[k] = model
        
        predictions = model.transform(cluster_prepared)
        silhouette = evaluator.evaluate(predictions)
        training_cost = model.summary.trainingCost

        k_results.append((k, float(silhouette), float(training_cost)))
        print(f"k = {k}, Silhouette = {silhouette:.4f}, Training Cost = {training_cost:.4f}")

        # Thu thập toàn bộ tập dữ liệu đã phân cụm cho K=2, 3, 4, 5 nhằm vẽ biểu đồ phân tích Silhouette
        if k in [2, 3, 4, 5]:
            all_preds = predictions.select("features", "prediction").collect()
            X_sample = np.array([row["features"].toArray() for row in all_preds])
            labels_sample = np.array([row["prediction"] for row in all_preds])
            samples_by_k[k] = (X_sample, labels_sample)

    print(f"[KIỂM TRA] PySpark Silhouette K=3: {[r[1] for r in k_results if r[0]==3][0]:.4f}")
    return k_results, k_models, samples_by_k



# Chạy quá trình tối ưu hóa chọn K
k_results, k_models, samples_by_k = run_kmeans_optimization(cluster_prepared)
print("-> Huấn luyện và khảo sát K tối ưu hoàn tất!")


ConnectionRefusedError: [WinError 10061] No connection could be made because the target machine actively refused it

## Bước 6: Phân tích & Gán nhãn hồ sơ khách hàng với giá trị K tốt nhất
Chạy phân cụm chính thức với `BEST_K = 3`, thực hiện xếp hạng và gán nhãn phân khúc khách hàng động dựa trên kết quả trung bình RFM.

In [11]:
def analyze_best_k_profiles(spark, cluster_prepared, k_results, k_models, total_customers, best_k=3):
    """Phân tích đặc trưng các cụm cho BEST_K, xếp hạng và gán nhãn kinh doanh động."""
    
    # 1. Hiển thị bảng so sánh K
    values_sql = ", ".join([
        f"({int(k)}, {float(silhouette)}, {float(training_cost)})"
        for k, silhouette, training_cost in k_results
    ])
    k_result_df = spark.sql(f"""
        SELECT *
        FROM VALUES {values_sql}
        AS k_table(k, Silhouette, Training_Cost)
    """)
    print("\n===== BẢNG SO SÁNH K =====")
    k_result_df.orderBy("k").show(truncate=False)

    best_row = k_result_df.filter(col("k") == best_k).first()
    print("\n===== CHỌN BEST_K CHO PHÂN KHÚC RFM =====")
    print("BEST_K:", best_k)
    print(f"Silhouette Score của K={best_k}:", round(float(best_row["Silhouette"]), 4))

    # 2. Xây dựng hồ sơ (profile) trung bình của từng cụm khách hàng
    kmeans_model = k_models[best_k]
    cluster_predictions = kmeans_model.transform(cluster_prepared)
    cluster_predictions = cluster_predictions.cache()

    cluster_profiles = (
        cluster_predictions
        .groupBy("prediction")
        .agg(
            avg("Recency").alias("Avg_Recency"),
            avg("Frequency").alias("Avg_Frequency"),
            avg("Monetary").alias("Avg_Monetary")
        )
        .collect()
    )

    # Xếp hạng Recency tăng dần (thấp = tốt = rank nhỏ)
    sorted_by_r = sorted(cluster_profiles, key=lambda x: x["Avg_Recency"])
    r_ranks = {row["prediction"]: rank for rank, row in enumerate(sorted_by_r)}

    # Xếp hạng Frequency giảm dần (cao = tốt = rank nhỏ)
    sorted_by_f = sorted(cluster_profiles, key=lambda x: x["Avg_Frequency"], reverse=True)
    f_ranks = {row["prediction"]: rank for rank, row in enumerate(sorted_by_f)}

    # Xếp hạng Monetary giảm dần (cao = tốt = rank nhỏ)
    sorted_by_m = sorted(cluster_profiles, key=lambda x: x["Avg_Monetary"], reverse=True)
    m_ranks = {row["prediction"]: rank for rank, row in enumerate(sorted_by_m)}

    # Tính toán tổng rank tích lũy
    cluster_scores = []
    for row in cluster_profiles:
        c_id = row["prediction"]
        score = r_ranks[c_id] + f_ranks[c_id] + m_ranks[c_id]
        cluster_scores.append({
            "prediction": c_id,
            "Avg_Recency": row["Avg_Recency"],
            "Avg_Frequency": row["Avg_Frequency"],
            "Avg_Monetary": row["Avg_Monetary"],
            "R_Rank": r_ranks[c_id],
            "F_Rank": f_ranks[c_id],
            "M_Rank": m_ranks[c_id],
            "Total_Rank": score
        })

    # Sắp xếp các cụm theo Total_Rank tăng dần (tổng rank càng thấp = cụm càng chất lượng)
    sorted_scores = sorted(cluster_scores, key=lambda x: x["Total_Rank"])

    print("\n===== BẢNG XẾP HẠNG (RANK) CÁC CỤM THEO TIÊU CHÍ RFM =====")
    print(f"{'Cụm ID':<8} | {'Avg_Recency':<12} | {'Avg_Frequency':<13} | {'Avg_Monetary':<12} | {'R_Rank':<6} | {'F_Rank':<6} | {'M_Rank':<6} | {'Total_Rank':<10}")
    print("-" * 92)
    for item in cluster_scores:
        print(f"{item['prediction']:<8} | {item['Avg_Recency']:<12.2f} | {item['Avg_Frequency']:<13.2f} | {item['Avg_Monetary']:<12.2f} | {item['R_Rank']:<6} | {item['F_Rank']:<6} | {item['M_Rank']:<6} | {item['Total_Rank']:<10}")

    # Tạo biểu thức gán nhãn động dựa trên xếp hạng
    label_expr = when(col("prediction") == sorted_scores[0]["prediction"], "Khách VIP")
    if len(sorted_scores) > 2:
        for rank in range(1, len(sorted_scores) - 1):
            label_expr = label_expr.when(col("prediction") == sorted_scores[rank]["prediction"], f"Khách Tiềm Năng (Rank {rank})")
    label_expr = label_expr.otherwise("Khách Ít Hoạt Động")

    cluster_predictions = cluster_predictions.withColumn("Cluster_Label", label_expr)

    # Dictionary mapping nhãn cho tương thích với phần vẽ đồ thị
    cluster_labels = {}
    for rank, item in enumerate(sorted_scores):
        c_id = item["prediction"]
        if rank == 0:
            cluster_labels[c_id] = "Khách VIP"
        elif rank == len(sorted_scores) - 1:
            cluster_labels[c_id] = "Khách Ít Hoạt Động"
        else:
            cluster_labels[c_id] = f"Khách Tiềm Năng (Rank {rank})"

    print("\n===== DỮ LIỆU KHÁCH HÀNG SAU PHÂN CỤM (MẪU 10 DÒNG) =====")
    cluster_predictions.select(
        "Customer_ID", "Customer_Name", "Recency", "Frequency", "Monetary", "Avg_Profit", "Total_Quantity", "prediction", "Cluster_Label"
    ).show(10, truncate=False)

    # 3. Phân bổ số lượng khách hàng theo cụm
    cluster_size = (
        cluster_predictions
        .groupBy("prediction", "Cluster_Label")
        .agg(count("*").alias("Cluster_Size"))
        .withColumn("Rate_Percent", spark_round(col("Cluster_Size") / total_customers * 100, 2))
        .orderBy("prediction")
    )
    print("\n===== PHÂN BỔ SỐ LƯỢNG KHÁCH HÀNG THEO CỤM =====")
    cluster_size.show(truncate=False)

    # 4. Đặc trưng hành vi chi tiết của từng cụm
    cluster_profile = (
        cluster_predictions
        .groupBy("prediction", "Cluster_Label")
        .agg(
            count("*").alias("Cluster_Size"),
            spark_round(avg("Recency"), 2).alias("Avg_Recency"),
            spark_round(avg("Frequency"), 2).alias("Avg_Frequency"),
            spark_round(avg("Monetary"), 2).alias("Avg_Monetary"),
            spark_round(avg("Avg_Profit"), 2).alias("Avg_Profit"),
            spark_round(avg("Avg_Discount"), 4).alias("Avg_Discount"),
            spark_round(avg("Total_Quantity"), 2).alias("Avg_Total_Quantity")
        )
        .withColumn("Rate_Percent", spark_round(col("Cluster_Size") / total_customers * 100, 2))
        .orderBy("prediction")
    )
    print("\n===== ĐẶC TRƯNG TRUNG BÌNH CỦA TỪNG CỤM KHÁCH HÀNG =====")
    cluster_profile.show(truncate=False)

    # 5. Top 10 khách hàng tiêu biểu xếp theo Monetary giảm dần
    print("\n===== TOP 10 KHÁCH HÀNG TIÊU BIỂU THEO TỪNG CỤM (XẾP THEO MONETARY GIẢM DẦN) =====")
    for c_id in sorted(cluster_labels.keys()):
        label = cluster_labels[c_id]
        print(f"\n--- Nhóm {c_id}: {label} ---")
        (
            cluster_predictions
            .filter(col("prediction") == c_id)
            .select("Customer_ID", "Customer_Name", "Recency", "Frequency", "Monetary", "Avg_Profit", "Total_Quantity")
            .orderBy(col("Monetary").desc())
            .show(10, truncate=False)
        )

    return cluster_predictions, cluster_profile, cluster_labels


# ======================================================
# CÁC HÀM TRỰC QUAN HÓA (VISUALIZATION)
# ======================================================


# Gán nhãn kinh doanh động & Phân tích hồ sơ khách hàng với BEST_K
cluster_predictions, cluster_profile, cluster_labels = analyze_best_k_profiles(
    spark, cluster_prepared, k_results, k_models, total_customers, best_k=BEST_K
)
print(f"-> Phân tích hồ sơ khách hàng với K={BEST_K} hoàn tất!")


NameError: name 'k_results' is not defined

## Bước 7: Trực quan hóa kết quả phân cụm
Định nghĩa các hàm vẽ biểu đồ:

In [4]:
def plot_kmeans_evaluation(k_results, best_k, cluster_profile, output_path="spark_kmeans_evaluation.png"):
    """Vẽ biểu đồ Silhouette, Elbow (WSSSE) và phân bổ số lượng khách hàng theo cụm trong 1x3 subplots."""
    print("\n===== ĐANG TẠO CÁC BIỂU ĐỒ TRỰC QUAN HÓA... =====")
    try:
        if not HAS_VISUAL_LIBS:
            raise ImportError("Thiếu thư viện trực quan hóa (matplotlib, pandas, scikit-learn)")

        plt.style.use('default')
        fig, axes = plt.subplots(1, 3, figsize=(18, 5), dpi=300, facecolor='white')
        
        for ax in axes:
            ax.set_facecolor('white')
            ax.tick_params(colors='black')
            ax.xaxis.label.set_color('black')
            ax.yaxis.label.set_color('black')
            ax.title.set_color('black')
            
        ks = [item[0] for item in k_results]
        silhouettes = [item[1] for item in k_results]
        costs = [item[2] for item in k_results]
        
        # Biểu đồ 1: Silhouette Score theo K
        axes[0].plot(ks, silhouettes, marker='o', color='b', linestyle='-', linewidth=2)
        axes[0].set_title('Chỉ số Silhouette theo số cụm (K)', fontsize=12, fontweight='bold', color='black')
        axes[0].set_xlabel('Số lượng cụm (K)', fontsize=10, color='black')
        axes[0].set_ylabel('Điểm Silhouette', fontsize=10, color='black')
        axes[0].grid(True, linestyle='--', alpha=0.6, color='grey')
        axes[0].axvline(x=best_k, color='r', linestyle='--', label=f'K tốt nhất = {best_k}')
        legend0 = axes[0].legend(facecolor='white', edgecolor='grey')
        for text in legend0.get_texts():
            text.set_color('black')
        
        # Biểu đồ 2: Elbow Method (Training Cost) theo K
        axes[1].plot(ks, costs, marker='s', color='g', linestyle='-', linewidth=2)
        axes[1].set_title('Phương pháp Khuỷu tay (Elbow Method)', fontsize=12, fontweight='bold', color='black')
        axes[1].set_xlabel('Số lượng cụm (K)', fontsize=10, color='black')
        axes[1].set_ylabel('Chi phí huấn luyện (WSSSE)', fontsize=10, color='black')
        axes[1].grid(True, linestyle='--', alpha=0.6, color='grey')
        axes[1].axvline(x=best_k, color='r', linestyle='--', label=f'K tốt nhất = {best_k}')
        legend1 = axes[1].legend(facecolor='white', edgecolor='grey')
        for text in legend1.get_texts():
            text.set_color('black')

        # Biểu đồ 3: Phân phối số lượng khách hàng theo cụm (Bar chart)
        if hasattr(cluster_profile, "toPandas"):
            profile_pdf = cluster_profile.toPandas()
        else:
            profile_pdf = cluster_profile
        
        colors = []
        for label in profile_pdf['Cluster_Label']:
            if "VIP" in label:
                colors.append('#2e7d32')      # VIP -> xanh lá
            elif "Tiềm Năng" in label or "Tiềm năng" in label:
                colors.append('#1565c0')      # Tiềm Năng -> xanh dương
            else:
                colors.append('#ff5722')      # Ít HĐ -> đỏ cam

        axes[2].bar(profile_pdf['prediction'].astype(str), profile_pdf['Cluster_Size'], color=colors)
        axes[2].set_title('Phân bổ số lượng khách hàng theo cụm', fontsize=12, fontweight='bold', color='black')
        axes[2].set_xlabel('Mã cụm (Cluster ID)', fontsize=10, color='black')
        axes[2].set_ylabel('Số lượng khách hàng', fontsize=10, color='black')
        axes[2].grid(True, axis='y', linestyle='--', alpha=0.6, color='grey')
        
        axes[2].set_ylim(0, max(profile_pdf['Cluster_Size']) * 1.15)
        
        for i, row in profile_pdf.iterrows():
            label_text = f"{row['Cluster_Label']}\n({row['Rate_Percent']}%)"
            axes[2].text(
                i, 
                row['Cluster_Size'] + (max(profile_pdf['Cluster_Size']) * 0.02), 
                label_text, 
                ha='center', 
                va='bottom', 
                fontweight='bold',
                fontsize=9,
                color='black'
            )

        plt.tight_layout()
        plt.savefig(output_path, dpi=300, facecolor='white', edgecolor='none')
        plt.close()
        print(f"-> Đã lưu biểu đồ tổng hợp tại: {os.path.abspath(output_path)}")
    except Exception as e:
        print(f"Lỗi khi vẽ biểu đồ Bước 17: {str(e)}")


def plot_rfm_radar(cluster_profile, output_path="rfm_radar_chart.png"):
    """Vẽ biểu đồ Radar so sánh tương quan đặc trưng RFM đã chuẩn hóa và đảo ngược Recency."""
    print("\n===== ĐANG VẼ BIỂU ĐỒ RADAR SO SÁNH ĐẶC TRƯNG RFM... =====")
    try:
        if not HAS_VISUAL_LIBS:
            raise ImportError("Thiếu thư viện trực quan hóa (matplotlib, pandas, scikit-learn)")

        plt.style.use('default')
        if hasattr(cluster_profile, "toPandas"):
            radar_df = cluster_profile.toPandas()
        else:
            radar_df = cluster_profile

        # Chuẩn hóa Min-Max cho từng trục Recency, Frequency, Monetary về khoảng [0.15, 1.0]
        # Việc đặt cận dưới là 0.15 giúp cụm thấp nhất không bị hội tụ về tâm (r=0) và biến mất
        for c_col in ['Avg_Recency', 'Avg_Frequency', 'Avg_Monetary']:
            min_val = radar_df[c_col].min()
            max_val = radar_df[c_col].max()
            if max_val != min_val:
                norm_val = (radar_df[c_col] - min_val) / (max_val - min_val)
                radar_df[c_col + '_norm'] = 0.15 + 0.85 * norm_val
            else:
                radar_df[c_col + '_norm'] = 0.5

        # Đảo ngược cột Recency sau khi chuẩn hóa (đối với khoảng [0.15, 1.0], ta lấy 1.15 - value)
        # Chú thích: Recency càng thấp (số ngày kể từ lần mua cuối càng nhỏ) thì khách hàng mua càng gần đây, 
        # tức là càng tốt. Do đó ta đảo ngược để giá trị chuẩn hóa cao (gần 1.0) biểu diễn mức độ tốt hơn.
        radar_df['Avg_Recency_norm'] = 1.15 - radar_df['Avg_Recency_norm']

        # Cấu hình các trục radar với nhãn rút gọn và rõ ràng hơn
        categories = [
            "Gần đây\n(Recency đảo)",
            "Tần suất\n(Frequency)",
            "Giá trị mua\n(Monetary)"
        ]
        N = len(categories)

        # Tính góc cho mỗi trục (3 trục)
        angles = [n / float(N) * 2 * np.pi for n in range(N)]
        angles += angles[:1]  # Khép kín vòng tròn bằng cách lặp lại góc đầu tiên

        # Khởi tạo biểu đồ radar với kích thước lớn hơn để tránh đè chữ (8x8 inches)
        fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True), dpi=300, facecolor='white')
        ax.set_facecolor('white')
        ax.tick_params(colors='black')
        ax.spines['polar'].set_color('black')
        
        # Đặt góc xoay xuất phát hướng thẳng đứng lên trên (90 độ)
        ax.set_theta_offset(np.pi / 2)
        ax.set_theta_direction(-1)

        # Đặt nhãn cho từng trục và thêm khoảng đệm (pad) tránh nhãn đè vào đường tròn ngoài cùng
        plt.xticks(angles[:-1], categories, color='black', size=10, fontweight='bold')
        ax.tick_params(pad=15)

        # Đặt nhãn và giới hạn cho vòng tròn đồng tâm (0 đến 1)
        ax.set_rlabel_position(0)
        plt.yticks([0.2, 0.4, 0.6, 0.8, 1.0], ["0.2", "0.4", "0.6", "0.8", "1.0"], color="grey", size=8)
        plt.ylim(0, 1)

        # Vẽ và tô màu cho từng cụm khách hàng
        for idx, row in radar_df.iterrows():
            values = [row['Avg_Recency_norm'], row['Avg_Frequency_norm'], row['Avg_Monetary_norm']]
            values += values[:1]  # Lặp lại giá trị đầu để khép kín hình
            
            label = row['Cluster_Label']
            if "VIP" in label:
                color = '#2e7d32'       # Xanh lá
            elif "Tiềm Năng" in label or "Tiềm năng" in label:
                color = '#1565c0'       # Xanh dương
            else:
                color = '#ff5722'       # Đỏ cam

            ax.plot(angles, values, linewidth=2, linestyle='solid', label=label, color=color)
            ax.fill(angles, values, color=color, alpha=0.2)

        # Cấu hình Legend nằm ở phía dưới biểu đồ, căn giữa và chia thành 3 cột
        legend = plt.legend(
            loc='upper center', 
            bbox_to_anchor=(0.5, -0.12), 
            ncol=3, 
            title="Nhóm khách hàng", 
            title_fontsize='10', 
            fontsize='9',
            frameon=True,
            facecolor='white',
            edgecolor='grey'
        )
        legend.get_title().set_color('black')
        for text in legend.get_texts():
            text.set_color('black')
        
        # Thêm tiêu đề cho biểu đồ radar với khoảng đệm lớn để không đè chữ trục trên cùng
        plt.title("So sánh đặc trưng RFM theo nhóm khách hàng", size=14, fontweight='bold', pad=30, color='black')
        
        plt.savefig(output_path, dpi=300, bbox_inches='tight', facecolor='white', edgecolor='none')
        plt.close()
        print(f"-> Đã lưu biểu đồ Radar tại: {os.path.abspath(output_path)}")
    except Exception as e:
        print(f"Lỗi khi vẽ biểu đồ Radar Bước 17.2: {str(e)}")


def plot_profile_heatmap(cluster_profile, output_path="rfm_profile_heatmap.png"):
    """Vẽ bảng tổng kết đặc trưng hành vi dạng Heatmap bằng màu sắc từ Red-Yellow-Green."""
    print("\n===== ĐANG VẼ BẢNG TỔNG KẾT ĐẶC TRƯNG DẠNG HEATMAP... =====")
    try:
        if not HAS_VISUAL_LIBS:
            raise ImportError("Thiếu thư viện trực quan hóa (matplotlib, pandas, scikit-learn)")

        plt.style.use('default')
        if hasattr(cluster_profile, "toPandas"):
            heatmap_df = cluster_profile.toPandas()
        else:
            heatmap_df = cluster_profile
        headers = ['Cluster', 'Nhãn', 'Avg_Recency', 'Avg_Frequency', 'Avg_Monetary', 'Avg_Profit', 'Rate_%']
        
        cell_text = []
        cell_colors = []

        min_max = {}
        num_cols = ['Avg_Recency', 'Avg_Frequency', 'Avg_Monetary', 'Avg_Profit', 'Rate_Percent']
        for col_name in num_cols:
            min_max[col_name] = {
                'min': heatmap_df[col_name].min(),
                'max': heatmap_df[col_name].max()
            }

        cmap = plt.get_cmap('RdYlGn')

        for idx, row in heatmap_df.iterrows():
            row_text = [
                str(int(row['prediction'])),
                str(row['Cluster_Label']),
                f"{row['Avg_Recency']:,.2f}",
                f"{row['Avg_Frequency']:,.2f}",
                f"{row['Avg_Monetary']:,.2f}",
                f"{row['Avg_Profit']:,.2f}",
                f"{row['Rate_Percent']}%"
            ]
            cell_text.append(row_text)

            row_colors = ['#f2f2f2', '#f2f2f2']

            # Cột Avg_Recency (thấp = tốt = xanh lá = 1.0; cao = kém = đỏ = 0.0)
            r_val = row['Avg_Recency']
            r_min = min_max['Avg_Recency']['min']
            r_max = min_max['Avg_Recency']['max']
            norm_r = (r_max - r_val) / (r_max - r_min) if r_max != r_min else 0.5
            row_colors.append(mcolors.to_hex(cmap(norm_r * 0.8 + 0.1)))

            # Cột Avg_Frequency (cao = tốt = xanh lá = 1.0)
            f_val = row['Avg_Frequency']
            f_min = min_max['Avg_Frequency']['min']
            f_max = min_max['Avg_Frequency']['max']
            norm_f = (f_val - f_min) / (f_max - f_min) if f_max != f_min else 0.5
            row_colors.append(mcolors.to_hex(cmap(norm_f * 0.8 + 0.1)))

            # Cột Avg_Monetary (cao = tốt = xanh lá = 1.0)
            m_val = row['Avg_Monetary']
            m_min = min_max['Avg_Monetary']['min']
            m_max = min_max['Avg_Monetary']['max']
            norm_m = (m_val - m_min) / (m_max - m_min) if m_max != m_min else 0.5
            row_colors.append(mcolors.to_hex(cmap(norm_m * 0.8 + 0.1)))

            # Cột Avg_Profit (cao = tốt = xanh lá = 1.0)
            p_val = row['Avg_Profit']
            p_min = min_max['Avg_Profit']['min']
            p_max = min_max['Avg_Profit']['max']
            norm_p = (p_val - p_min) / (p_max - p_min) if p_max != p_min else 0.5
            row_colors.append(mcolors.to_hex(cmap(norm_p * 0.8 + 0.1)))

            # Cột Rate_Percent (tỷ lệ % phân bổ)
            rate_val = row['Rate_Percent']
            rate_min = min_max['Rate_Percent']['min']
            rate_max = min_max['Rate_Percent']['max']
            norm_rate = (rate_val - rate_min) / (rate_max - rate_min) if rate_max != rate_min else 0.5
            row_colors.append(mcolors.to_hex(cmap(norm_rate * 0.8 + 0.1)))

            cell_colors.append(row_colors)

        fig, ax = plt.subplots(figsize=(10, 4), dpi=300, facecolor='white')
        ax.set_facecolor('white')
        ax.axis('off')

        table = ax.table(
            cellText=cell_text,
            cellColours=cell_colors,
            colLabels=headers,
            loc='center',
            cellLoc='center'
        )

        table.auto_set_font_size(False)
        table.set_fontsize(10)
        table.scale(1.2, 2.2)

        header_color = '#34495e'
        for col_idx in range(len(headers)):
            cell = table[0, col_idx]
            cell.set_facecolor(header_color)
            cell.set_text_props(color='white', weight='bold', fontsize=10)

        # Force all data cell text color inside table to be black (except header) to prevent invisible white text
        for (row_idx, col_idx), cell in table.get_celld().items():
            if row_idx > 0:
                cell.set_text_props(color='black')

        plt.title("Bảng tổng kết đặc trưng từng nhóm khách hàng", fontsize=14, fontweight='bold', pad=25, color='black')
        plt.tight_layout()
        plt.savefig(output_path, dpi=300, facecolor='white', edgecolor='none')
        plt.close()
        print(f"-> Đã lưu bảng tổng kết tại: {os.path.abspath(output_path)}")
    except Exception as e:
        print(f"Lỗi khi vẽ bảng tổng kết Bước 17.3: {str(e)}")


def plot_silhouette_profiles(samples_by_k, output_path="spark_silhouette_analysis_profiles.png"):
    """Vẽ chi tiết biểu đồ mật độ Silhouette Coefficient cho K = 2, 3, 4, 5 (Layout 2x2)."""
    print("\n===== ĐANG VẼ BIỂU ĐỒ SILHOUETTE COEFFICIENT CHO TỪNG PHÂN CỤM (K=2,3,4,5)... =====")
    try:
        if not HAS_VISUAL_LIBS:
            raise ImportError("Thiếu thư viện trực quan hóa (matplotlib, pandas, scikit-learn)")

        plt.style.use('default')
        fig, axes = plt.subplots(2, 2, figsize=(16, 12), facecolor='white')
        axes_flat = axes.flatten()
        
        for idx, k in enumerate([2, 3, 4, 5]):
            ax = axes_flat[idx]
            ax.set_facecolor('white')
            ax.tick_params(colors='black')
            ax.xaxis.label.set_color('black')
            ax.yaxis.label.set_color('black')
            ax.title.set_color('black')
            
            X_sample, labels_sample = samples_by_k[k]
            
            # Tính toán chỉ số Silhouette chi tiết cho từng mẫu dựa trên Metric sqeuclidean để khớp với PySpark
            silhouette_avg = silhouette_score(X_sample, labels_sample, metric="sqeuclidean")
            sample_silhouette_values = silhouette_samples(X_sample, labels_sample, metric="sqeuclidean")
            
            if k == 3:
                print(f"[KIỂM TRA] sklearn Silhouette K=3: {silhouette_avg:.4f}")
            
            ax.set_xlim([-0.1, 1])
            ax.set_ylim([0, len(X_sample) + (k + 1) * 10])
            
            y_lower = 10
            for i in range(k):
                ith_cluster_silhouette_values = sample_silhouette_values[labels_sample == i]
                ith_cluster_silhouette_values.sort()
                
                size_cluster_i = ith_cluster_silhouette_values.shape[0]
                y_upper = y_lower + size_cluster_i
                
                color = cm.nipy_spectral(float(i) / k)
                ax.fill_betweenx(
                    np.arange(y_lower, y_upper),
                    0,
                    ith_cluster_silhouette_values,
                    facecolor=color,
                    edgecolor=color,
                    alpha=0.7
                )
                
                ax.text(-0.05, y_lower + 0.5 * size_cluster_i, str(i), fontweight='bold', color='black')
                y_lower = y_upper + 10
                
            ax.set_title(f"Silhouette Plot for KMeans with K = {k}\n(Average Score: {silhouette_avg:.4f})", fontsize=11, fontweight='bold', color='black')
            ax.set_xlabel("Silhouette coefficient values", color='black')
            ax.set_ylabel("Cluster label", color='black')
            
            ax.axvline(x=silhouette_avg, color="red", linestyle="--", label=f"Average: {silhouette_avg:.4f}")
            legend = ax.legend(loc="upper right", fontsize=8, facecolor='white', edgecolor='grey')
            for text in legend.get_texts():
                text.set_color('black')
            
            ax.set_yticks([])
            ax.set_xticks([-0.1, 0, 0.2, 0.4, 0.6, 0.8, 1])
            ax.grid(True, linestyle='--', alpha=0.5, color='grey')
            
        plt.tight_layout()
        plt.savefig(output_path, dpi=300, facecolor='white', edgecolor='none')
        plt.close()
        print(f"-> Đã lưu biểu đồ phân tích Silhouette chi tiết tại: {os.path.abspath(output_path)}")
    except Exception as e:
        print(f"Lỗi khi vẽ biểu đồ phân tích Silhouette: {str(e)}")


# ======================================================
# LUỒNG THỰC THI CHÍNH
# ======================================================

print("-> Đã định nghĩa xong các hàm trực quan hóa!")


-> Đã định nghĩa xong các hàm trực quan hóa!


### Thực hiện vẽ và hiển thị các biểu đồ phân tích

In [3]:
# 1. Vẽ biểu đồ đánh giá KMeans (Elbow, Silhouette, Bar chart phân bổ)
plot_kmeans_evaluation(k_results, BEST_K, cluster_profile)

# 2. Vẽ biểu đồ Radar so sánh các chiều RFM
plot_rfm_radar(cluster_profile)

# 3. Vẽ biểu đồ Heatmap hồ sơ đặc trưng khách hàng
plot_profile_heatmap(cluster_profile)

# 4. Vẽ biểu đồ phân tích mật độ Silhouette Coefficient cho K=2,3,4,5
plot_silhouette_profiles(samples_by_k)

print("-> Đã vẽ xong và lưu toàn bộ biểu đồ phân tích!")


NameError: name 'k_results' is not defined

## Bước 8: Giải phóng bộ nhớ & Dừng Spark Session

In [ ]:
# Dừng Spark session
print("\n===== ĐANG DỪNG SPARK CLUSTER =====")
spark.stop()
print("-> Đã dừng Spark Session thành công!")
